In [1]:


# Core imports for the whole lab
import numpy as np
import sympy as sp

x, y = sp.symbols('x y')      # symbolic variables we'll reuse
sp.init_printing()            # pretty-print symbolic math
np.random.seed(42)
print('Setup complete. SymPy', sp.__version__, '| NumPy', np.__version__)



Setup complete. SymPy 1.14.0 | NumPy 2.0.2


In [2]:

# The derivative is the slope: how much f changes for a tiny step h
def f(x):
    return x**2

h = 1e-6
slope_at_3 = (f(3 + h) - f(3)) / h
print('Numerical f\'(3):', round(slope_at_3, 4), ' (exact = 6)')


Numerical f'(3): 6.0  (exact = 6)


In [3]:
expr = x**2
deriv = sp.diff(expr, x)          # differentiate w.r.t. x
print('d/dx (x**2) =', deriv)     # 2*x

# Evaluate the symbolic derivative at x = 3
print('Symbolic f\'(3) =', deriv.subs(x, 3))

d/dx (x**2) = 2*x
Symbolic f'(3) = 6


In [5]:
def g(x):
    return x**3 + 2*x

# 1. Numerical derivative at x = 2 (use h = 1e-6)
# YOUR CODE HERE
h = 1e-6
x = 2

numerical = (g(x + h) - g(x)) / h

print("Numerical derivative:", numerical)

# 2. Symbolic derivative of x**3 + 2*x
# YOUR CODE HERE
x = sp.symbols('x')

derivative = sp.diff(x**3 + 2*x, x)

print("Symbolic derivative:", derivative)

# 3. Evaluate the symbolic derivative at x = 2 and compare
# YOUR CODE HERE
symbolic_value = derivative.subs(x, 2)

print("Symbolic derivative at x = 2:", symbolic_value)

print("Do they match?", np.allclose(numerical, float(symbolic_value)))


Numerical derivative: 14.000006002490295
Symbolic derivative: 3*x**2 + 2
Symbolic derivative at x = 2: 14
Do they match? True


In [6]:
f2 = x**2 + 3*x*y + y**2

# A partial derivative differentiates ONE variable, holding others fixed
print('df/dx =', sp.diff(f2, x))     # 2*x + 3*y
print('df/dy =', sp.diff(f2, y))     # 3*x + 2*y



df/dx = 2*x + 3*y
df/dy = 3*x + 2*y


In [7]:
# The gradient stacks every partial derivative into one vector
grad = [sp.diff(f2, v) for v in (x, y)]
print('grad f =', grad)

# Evaluate the gradient at the point (x=1, y=2)
grad_at = [g.subs({x: 1, y: 2}) for g in grad]
print('grad f at (1, 2) =', grad_at)   # points in the steepest-ascent direction

grad f = [2*x + 3*y, 3*x + 2*y]
grad f at (1, 2) = [8, 7]


In [8]:


h2 = x**2 * y + sp.sin(y)

# 1. dh/dx and dh/dy
# YOUR CODE HERE
dh_dx = sp.diff(h2, x)
dh_dy = sp.diff(h2, y)

print("dh/dx =", dh_dx)
print("dh/dy =", dh_dy)
# 2. Assemble the gradient list
# YOUR CODE HERE
gradient = [dh_dx, dh_dy]

print("Gradient =", gradient)

# 3. Evaluate at (x=2, y=0)  -> hint: .subs({x: 2, y: 0})
# YOUR CODE HERE
gradient_at_point = [g.subs({x: 2, y: 0}) for g in gradient]

print("Gradient at (2,0) =", gradient_at_point)



dh/dx = 2*x*y
dh/dy = x**2 + cos(y)
Gradient = [2*x*y, x**2 + cos(y)]
Gradient at (2,0) = [0, 5]


In [9]:
# y = sin(x**2) is a composition: outer = sin(u), inner = u = x**2
# Chain rule:  dy/dx = cos(u) * du/dx = cos(x**2) * 2x
by_hand = sp.cos(x**2) * 2*x
by_sympy = sp.diff(sp.sin(x**2), x)

print('By hand :', by_hand)
print('By SymPy:', by_sympy)
print('Match?  ', sp.simplify(by_hand - by_sympy) == 0)


By hand : 2*x*cos(x**2)
By SymPy: 2*x*cos(x**2)
Match?   True


In [10]:
# y = (3x + 1)**4  -> outer^4, inner (3x+1)
expr3 = (3*x + 1)**4
print('d/dx (3x+1)^4 =', sp.diff(expr3, x))   # 12*(3x+1)^3



d/dx (3x+1)^4 = 12*(3*x + 1)**3


In [11]:
# y = exp(x**2 + 1);  inner = x**2 + 1, inner' = 2x, outer' = exp(inner)

# 1. By hand (as a SymPy expression)
# YOUR CODE HERE
y_by_hand = sp.exp(x**2 + 1) * (2*x)

print("By hand:", y_by_hand)
# 2. With sp.diff
# YOUR CODE HERE
y_by_sympy = sp.diff(sp.exp(x**2 + 1), x)

print("By SymPy:", y_by_sympy)

# 3. Confirm they match
# YOUR CODE HERE
print("Do they match?", sp.simplify(y_by_hand - y_by_sympy) == 0)

By hand: 2*x*exp(x**2 + 1)
By SymPy: 2*x*exp(x**2 + 1)
Do they match? True


In [12]:
# Tiny toy problem: 4 samples, 3 input features, 5 hidden units, 1 output
X = np.random.randn(4, 3)
Y = np.random.randn(4, 1)
W1 = np.random.randn(3, 5) * 0.1
W2 = np.random.randn(5, 1) * 0.1

z1 = X @ W1                 # linear layer 1
h  = np.maximum(0, z1)      # ReLU activation
y_hat = h @ W2              # linear layer 2 (prediction)
loss = ((y_hat - Y) ** 2).mean()
print('Initial loss:', round(loss, 4))



Initial loss: 1.6936


In [13]:
# Work backwards from the loss, one link at a time
dy   = 2 * (y_hat - Y) / Y.size      # d loss / d y_hat
dW2  = h.T @ dy                      # d loss / d W2
dh   = dy @ W2.T                     # d loss / d h
dz1  = dh * (z1 > 0)                 # ReLU gradient (1 where z1>0 else 0)
dW1  = X.T @ dz1                     # d loss / d W1

print('dW1 shape:', dW1.shape, '(matches W1)')
print('dW2 shape:', dW2.shape, '(matches W2)')


dW1 shape: (3, 5) (matches W1)
dW2 shape: (5, 1) (matches W2)


In [14]:
lr = 0.1
W1 -= lr * dW1               # step downhill
W2 -= lr * dW2

# Recompute the loss after the update
h_new = np.maximum(0, X @ W1)
loss_new = ((h_new @ W2 - Y) ** 2).mean()
print('Loss before:', round(loss, 4))
print('Loss after :', round(loss_new, 4), '-> should be lower')


Loss before: 1.6936
Loss after : 1.6524 -> should be lower


In [16]:
Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)
Wa = np.random.randn(4, 8) * 0.1     # input -> hidden
Wb = np.random.randn(8, 1) * 0.1     # hidden -> output

# 1. Forward pass: z1, h = ReLU(z1), y_hat, loss
# YOUR CODE HERE
z1 = Xb @ Wa
h = np.maximum(0, z1)      # ReLU
y_hat = h @ Wb

loss = np.mean((y_hat - Yb) ** 2)

print("Loss before:", loss)

# 2. Backward pass: dy, dWb, dh, dz1, dWa
# YOUR CODE HERE
dy = (2 / len(Yb)) * (y_hat - Yb)
dWb = h.T @ dy

dh = dy @ Wb.T
dz1 = dh * (z1 > 0)

dWa = Xb.T @ dz1

# 3. One step with lr = 0.05; print loss before and after
# YOUR CODE HERE
lr = 0.05

Wa = Wa - lr * dWa
Wb = Wb - lr * dWb

# Forward pass again
z1 = Xb @ Wa
h = np.maximum(0, z1)
y_hat = h @ Wb

new_loss = np.mean((y_hat - Yb) ** 2)

print("Loss after:", new_loss)


Loss before: 0.9012858906993045
Loss after: 0.8998731831845102


In [17]:
f5 = x**2 + 3*x*y + y**2
H = sp.hessian(f5, (x, y))
print('Hessian of f:')
sp.pprint(H)        # [[2, 3], [3, 2]]

Hessian of f:
⎡2  3⎤
⎢    ⎥
⎣3  2⎦


In [18]:
# Minimise f(x) = (x - 4)**2, whose minimum is at x = 4.
# Update rule:  x <- x - lr * f'(x),  with f'(x) = 2*(x - 4)
xv = 0.0          # starting point
lr = 0.2
for step in range(15):
    grad = 2 * (xv - 4)        # the derivative
    xv = xv - lr * grad        # step against the gradient
print('Converged x:', round(xv, 3), ' (true minimum = 4)')


Converged x: 3.998  (true minimum = 4)


In [19]:
# 1. Hessian of x**4 + y**2
# YOUR CODE HERE
H = sp.hessian(x**4 + y**2, (x, y))

print("Hessian Matrix:")
print(H)

# 2. Gradient descent to minimise (x - 7)**2
xv = 0.0
lr = 0.1

for step in range(20):
    grad = 2 * (xv - 7)
    xv = xv - lr * grad

print("Final x:", xv)



Hessian Matrix:
Matrix([[12*x**2, 0], [0, 2]])
Final x: 6.91929549467752
